In [ ]:
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

BASE = Path("/kaggle/input")
comp = next(p.parent for p in BASE.rglob("sample_submission.csv"))
print("dataset:", comp)

train_dir = comp / "train"
val_dir = comp / "val"
test_dir = comp / "test"


In [ ]:
def load_gt(path):
    df = pd.read_csv(path)
    return df.rename(columns={df.columns[1]: "classe"})

train = load_gt(train_dir / "ground_truth.csv")
val = load_gt(val_dir / "ground_truth.csv")
sub = pd.read_csv(comp / "sample_submission.csv")

print(train.shape, val.shape, sub.shape)
train.head()


In [ ]:
def balance(df, name):
    c = df["classe"].value_counts().sort_index()
    pos = c.get(1, 0) / len(df)
    print(f"{name:5} n={len(df):5}  authentic={c.get(0,0):5}  manipulated={c.get(1,0):5}  pos_rate={pos:.3f}")
    return c

bt = balance(train, "train")
bv = balance(val, "val")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, c, name in zip(ax, [bt, bv], ["train", "val"]):
    a.bar(["authentic", "manipulated"], [c.get(0, 0), c.get(1, 0)], color=["#4c72b0", "#c44e52"])
    a.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
train_files = {p.name for p in (train_dir / "images").iterdir()}
val_files = {p.name for p in (val_dir / "images").iterdir()}
test_files = {p.name for p in (test_dir / "images").iterdir()}

print("images sur disque:", len(train_files), len(val_files), len(test_files))
print("labels manquants train:", len(set(train.image_id) - train_files))
print("labels manquants val:", len(set(val.image_id) - val_files))
print("test == sample_submission:", test_files == set(sub.image_id))
print("chevauchement train/val:", len(train_files & val_files))


In [ ]:
def scan(img_dir, n=800):
    files = sorted(p.name for p in img_dir.iterdir())
    step = max(1, len(files) // n)
    rows = []
    for name in files[::step]:
        with Image.open(img_dir / name) as im:
            rows.append((im.width, im.height, im.mode))
    return pd.DataFrame(rows, columns=["w", "h", "mode"])

dims = scan(train_dir / "images")
print(dims["mode"].value_counts().to_dict())
dims[["w", "h"]].describe()


In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(dims.w, dims.h, s=6, alpha=0.3)
plt.xlabel("largeur"); plt.ylabel("hauteur"); plt.title("résolutions (échantillon train)")
plt.show()


In [ ]:
files = sorted(p.name for p in (train_dir / "images").iterdir())
step = max(1, len(files) // 800)
sample_ids = files[::step]
lbl = train.set_index("image_id")["classe"]

sizes = []
for name in sample_ids:
    with Image.open(train_dir / "images" / name) as im:
        sizes.append((name, im.width * im.height, lbl.get(name, -1)))

s = pd.DataFrame(sizes, columns=["image_id", "pixels", "classe"])
s.groupby("classe")["pixels"].describe()[["mean", "min", "max"]]


In [ ]:
def show(cls, k=4):
    ids = train[train.classe == cls].image_id.head(k)
    fig, ax = plt.subplots(1, k, figsize=(3 * k, 3))
    for a, name in zip(ax, ids):
        a.imshow(Image.open(train_dir / "images" / name))
        a.axis("off")
    fig.suptitle("authentic" if cls == 0 else "manipulated")
    plt.show()

show(0)
show(1)
